# Fase 2: RAG

## 1. Chatbot básico

### 1.1. Configuración del cliente de Groq

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
API_KEY = os.getenv("GROQ_KEY")

client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=API_KEY)

### 1.2. Comparar variantes de system prompt

In [3]:
SYSTEM_PROMPTS = {
    "Asistente General": (
        "Eres un asistente útil que responde preguntas de forma clara y comprensible. "
        "Proporciona respuestas correctas y breves. "
        "No es necesario usar un lenguaje técnico."
    ),
    "Asistente Especializado": (
        "Eres un asistente experto en privacidad digital, protección de datos y términos de servicio. "
        "Responde de forma breve, precisa, clara y estructurada. "
        "Utiliza lenguaje técnico cuando sea necesario, pero mantén la claridad. "
    ),
    "Asistente Restrictivo": (
        "Eres un asistente especializado en privacidad, protección de datos y términos de servicio."
        "Responde únicamente a preguntas relacionadas con estos temas."
        "Si la pregunta no está relacionada con privacidad, protección de datos o términos digitales, debes responder exactamente:"
        "\"No dispongo de información suficiente para responder con precisión.\""
        "No respondas preguntas fuera de este dominio bajo ninguna circunstancia."
        "Usa un tono formal y respuestas breves y estructuradas."
    ),
}

PREGUNTAS = [
    # Preguntas del dominio
    "¿Qué son las cookies y para qué se utilizan?",
    "¿Qué derechos tiene un usuario según el RGPD?",
    "¿Qué datos personales recopila Spotify y con qué finalidad?",

    # Preguntas fuera del dominio (para analizar alucinaciones y límites del modelo)
    "¿Cómo funciona la fotosíntesis en las plantas?",
    "¿Cómo se relacionan la teoría de la relatividad y la mecánica cuántica en la descripción del espacio-tiempo?"
]

In [6]:
MODEL = "llama-3.1-8b-instant"

def generar_respuesta(pregunta, system_prompt, temperature=1.0):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": pregunta}
        ],
        max_completion_tokens=300,
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

In [37]:
# Comparar respuestas de los tres asistentes para cada pregunta
for pregunta in PREGUNTAS:
    print(f"\nPregunta: {pregunta}\n{'='*50}")
    for asistente, system_prompt in SYSTEM_PROMPTS.items():
        respuesta = generar_respuesta(pregunta, system_prompt)
        print(f"{asistente}:\n{respuesta}\n{'-'*50}")


Pregunta: ¿Cómo se relacionan la teoría de la relatividad y la mecánica cuántica en la descripción del espacio-tiempo?
Asistente General:
La teoría de la relatividad y la mecánica cuántica son dos marcos teóricos fundamentales de la física que se relacionan en la descripción del espacio-tiempo de manera importante.

La teoría de la relatividad general, desarrollada por Albert Einstein, describe el espacio-tiempo como una sola entidad curva y dinámica, influenciada por la presencia de masa y energía. Esto implica que el espacio y el tiempo no son absolutos, sino que están interconectados y se curvan en presencia de objetos masivos.

Por otro lado, la mecánica cuántica describe los comportamientos de los objetos a nivel atómico y subatómico, donde las partículas se comportan de manera probabilística y no determinista. La mecánica cuántica también tiene un impacto en la descripción del espacio-tiempo, ya que sugiere que el espacio-t
--------------------------------------------------
Asis

### 1.3. Comparar parámetros de generación

In [11]:
# Se selecciona una pregunta que permita evaluar precisión + control + alucinaciones
pregunta = PREGUNTAS[2]
system_prompt = SYSTEM_PROMPTS["Asistente Restrictivo"]

print("Temperatura = 1.0 (respuestas más creativas y variadas)\n")
print(generar_respuesta(pregunta, system_prompt, temperature=1.0))

print(f"\n{'='*50}\nTemperatura = 0.6 (respuestas más coherentes y menos creativas)\n")
print(generar_respuesta(pregunta, system_prompt, temperature=0.6))

print(f"\n{'='*50}\nTemperatura = 0.2 (respuestas más precisas y menos creativas)\n")
print(generar_respuesta(pregunta, system_prompt, temperature=0.2))

Temperatura = 1.0 (respuestas más creativas y variadas)

Spotify recopila diversos tipos de datos personales, incluyendo:

1. **Información de cuenta**:nombre de usuario, contraseña, correo electrónico, fecha de nacimiento y género.
2. **Datos de actividad**: reproducciones de canciones, listas de reproducción, búsqueda de contenidos y duración de la sesión.
3. **Información de dispositivo**: tipo de dispositivo, idioma, versión de sistema operativo y datos de localización.
4. **Datos de pago**: información de tarjeta de crédito o débito, método de pago y saldo.

El fin principal de recopilar estos datos es proporcionar servicios a los usuarios, como:

- Personalizar recomendaciones de música.
- Permitir el acceso a contenido premium.
- Mejorar la experiencia de usuario y el rendimiento del servicio.
- Cumplir con obligaciones legales y regulaciones, como las relacionadas con la protección de menores.

No obstante, Spotify ha publicado una política de privacidad detallada, disponible e

 **Conclusión:** La mejor combinación para RAG es "Asistente Restrictivo" como system prompt + temperature=0.2

## 2. Chunking + embeddings

En este apartado se compararán los métodos de chunking por tamaño fijo y recursivo mostrando como quedan los chunks y evaluando cualitativamente la calidad del retrieval. 

Para ello, se usará solo uno de los documentos del corpus. 

Una vez seleccionada la estrategia de chunking más óptima, se implementará en el apartado 3 el pipeline completo almacenando los embeddings en una base de datos vectorial.

### 2.1. Comparar configuraciones de chunking 

In [51]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# Chunking por tamaño fijo
splitter_fijo = CharacterTextSplitter(
    separator="\n",
    chunk_size=500,      # máximo de caracteres por chunk
    chunk_overlap=50,    # solapamiento entre chunks consecutivos
    length_function=len
)

# Chunking recursivo
splitter_recursivo = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # máximo de caracteres por chunk
    chunk_overlap=150,   # 15% de solapamiento
    # Prueba separadores de mayor a menor granularidad
    separators=["\n# ", "\n## ", "\n### ", "\n---\n", "\n\n", "\n", ". ", " "],
    length_function=len,
)

In [52]:
# Comparar chunks generados por cada método
def mostrar_chunks(chunks):
    print(f"Chunks generados: {len(chunks)}\n")
    # Mostrar chunks del 20 al 25 para ver variedad
    for i, chunk in enumerate(chunks[19:25]):
        print(f"--- Chunk {i + 20} ({len(chunk)} chars) ---")
        print(chunk)
        print()

# Para comparar ambos metodos, se usa uno de los documentos del corpus
with open("corpus/guides/gdpr_explained.md", "r", encoding="utf-8") as f:
    texto = f.read()

# Generar chunks usando ambos métodos
chunks_fijo = splitter_fijo.split_text(texto)
chunks_recursivo = splitter_recursivo.split_text(texto)


Created a chunk of size 760, which is longer than the specified 500


In [53]:
print("Chunks generados por el método de tamaño fijo:")
mostrar_chunks(chunks_fijo)

Chunks generados por el método de tamaño fijo:
Chunks generados: 34

--- Chunk 20 (331 chars) ---
### Data protection by design and by default
From now on, everything you do in your organization must, “by design and by default,” consider data protection. Practically speaking, this means you must consider the data protection principles in the design of any new product or activity. The GDPR covers this principle in Article 25 .

--- Chunk 21 (297 chars) ---
Suppose, for example, you’re launching a new app for your company. You have to think about what personal data the app could possibly collect from users, then consider ways to minimize the amount of data and how you will secure it with the latest technology.
### When you’re allowed to process data

--- Chunk 22 (424 chars) ---
### When you’re allowed to process data
Article 6 lists the instances in which it’s legal to process person data. Don’t even think about touching somebody’s personal data — don’t collect it, don’t store it, don’t

In [54]:
print("Chunks generados por el método recursivo:")
mostrar_chunks(chunks_recursivo)

Chunks generados por el método recursivo:
Chunks generados: 26

--- Chunk 20 (279 chars) ---
Once you’ve determined the lawful basis for your data processing, you need to document this basis and notify the data subject (transparency!). And if you decide later to change your justification, you need to have a good reason, document this reason, and notify the data subject.

--- Chunk 21 (651 chars) ---
### Consent

There are strict new rules about what constitutes consent from a data subject to process their information.

- Consent must be “freely given, specific, informed and unambiguous.”
- Requests for consent must be “clearly distinguishable from the other matters” and presented in “clear and plain language.”
- Data subjects can withdraw previously given consent whenever they want, and you have to honor their decision. You can’t simply change the legal basis of the processing to one of the other justifications.
- Children under 13 can only give consent with permission from their paren

In [47]:
from sentence_transformers import SentenceTransformer

modelo_emb = SentenceTransformer("intfloat/multilingual-e5-base")

# Generar embeddings de chunks 
def generar_embeddings(chunks):
    list_chunks = ["passage: " + c for c in chunks]
    embeddings = modelo_emb.encode(
        list_chunks,
        normalize_embeddings=True
    )
    return embeddings

embeddings_chunks_fijo = generar_embeddings(chunks_fijo)
embeddings_chunks_recursivo = generar_embeddings(chunks_recursivo)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6620.50it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [55]:
# Evaluar cualitativamente la calidad del retrieval para cada método de chunking 
import numpy as np

QUERIES = [
    "What is the GDPR?",
    "Which are the data protection principles?",
    "What are the rights of a user according to the GDPR?",
]

def mostrar_similitudes(embeddings, chunks, queries, top_k=3):
    """Muestra los chunks más similares a cada query según los embeddings."""
    for query in queries:
        query_emb = modelo_emb.encode(["query: " + query], normalize_embeddings=True)[0]

        # Similitud coseno entre la query y todos los documentos
        similitudes = modelo_emb.similarity(query_emb, embeddings)[0].numpy()
        
        # Índices ordenados de mayor a menor similitud
        top_indices = np.argsort(similitudes)[-top_k:][::-1]

        print(f"\nQuery: {query}\n{'-'*50}")
        for i, idx in enumerate(top_indices):
            print(f"{i + 1}. (sim={similitudes[idx]:.3f}) {chunks[idx]}")


In [56]:
print("Similitudes para chunks generados por el método de tamaño fijo:")
mostrar_similitudes(embeddings_chunks_fijo, chunks_fijo, QUERIES)

Similitudes para chunks generados por el método de tamaño fijo:

Query: What is the GDPR?
--------------------------------------------------
1. (sim=0.907) The General Data Protection Regulation (GDPR) is the toughest privacy and security law in the world. Though it was drafted and passed by the European Union (EU), it imposes obligations onto organizations anywhere, so long as they target or collect data related to people in the EU. The regulation was put into effect on May 25, 2018. The GDPR will levy harsh fines against those who violate its privacy and security standards, with penalties reaching into the tens of millions of euros.
2. (sim=0.890) If you’ve found this page — “what is the GDPR?” — chances are you’re looking for a crash course. Maybe you haven’t even found the document itself yet (tip: here’s the full regulation ). Maybe you don’t have time to read the whole thing. This page is for you. In this article, we try to demystify the GDPR and, we hope, make it less overwhelmi

In [57]:
print("Similitudes para chunks generados por el método recursivo:")
mostrar_similitudes(embeddings_chunks_recursivo, chunks_recursivo, QUERIES)

Similitudes para chunks generados por el método recursivo:

Query: What is the GDPR?
--------------------------------------------------
1. (sim=0.907) The General Data Protection Regulation (GDPR) is the toughest privacy and security law in the world. Though it was drafted and passed by the European Union (EU), it imposes obligations onto organizations anywhere, so long as they target or collect data related to people in the EU. The regulation was put into effect on May 25, 2018. The GDPR will levy harsh fines against those who violate its privacy and security standards, with penalties reaching into the tens of millions of euros.
2. (sim=0.893) # What is GDPR, the EU’s new data protection law?

What is the GDPR? Europe’s new data privacy and security law includes hundreds of pages’ worth of new requirements for organizations around the world. This GDPR overview will help you understand the law and determine what parts of it apply to you.
3. (sim=0.881) With the GDPR, Europe is signalin

**Conclusiones:** El método recursivo genera chunks más coherentes y relevantes para las queries, lo que mejora la calidad del retrieval. El método de tamaño fijo produce chunks arbitrarios que pueden cortar información importante, resultando en similitudes más bajas y peor retrieval al devolver chunks menos relevantes.

Por ello, en el pipeline completo se aplicará chunking recursivo con 20% de solapamiento entre chunks.

## 3. Pipeline RAG completo

### 3.1. Chunking + Embeddings de todos los documentos del corpus

In [1]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunking_documento(doc_path):
    """
    Realiza chunking recursivo con solapamiento para un documento,
    devolviendo chunks junto con metadatos de archivo, categoria y empresa.
    """
    # Extraer nombre del archivo y categoría para metadatos
    ruta = Path(doc_path)
    archivo = ruta.name
    categoria = ruta.parent.name

    # Guardar empresa como metadato adicional
    nombre_archivo = archivo.lower()
    if "google" in nombre_archivo:
        empresa = "Google"
    elif "spotify" in nombre_archivo:
        empresa = "Spotify"
    else:
        empresa = None

    # Leer el contenido del documento
    with open(doc_path, "r", encoding="utf-8") as f:
        texto = f.read()

    # Chunking recursivo
    splitter_recursivo = RecursiveCharacterTextSplitter(
        chunk_size=1000,      # máximo de caracteres por chunk
        chunk_overlap=150,    # % de solapamiento
        # Prueba separadores de mayor a menor granularidad
        separators=["\n# ", "\n## ", "\n### ", "\n---", "\n\n", "\n", ". ", " "],
        length_function=len,
    )
    chunks = splitter_recursivo.split_text(texto)

    # En algunos casos, los titulos quedan aislados como chunks muy cortos. 
    # Se unen al siguiente chunk para mejorar contexto.
    chunks_unidos = []
    buffer = ""
    for chunk in chunks:
        # si es título (empieza por #) y chunk corto
        if chunk.startswith("#") and len(chunk) < 100:
            buffer = chunk
        else:
            if buffer:
                chunks_unidos.append(buffer + "\n" + chunk)
                buffer = ""
            else:
                chunks_unidos.append(chunk)
    if buffer:
        chunks_unidos.append(buffer)

    # Devolver chunks + metadatos
    return {
        "archivo": archivo,
        "categoria": categoria,
        "empresa": empresa,
        "chunks": chunks_unidos,
    }


def chunking_corpus(corpus_dir):
    """
    Realiza el proceso de chunking para todos los documentos .md en un directorio dado, 
    devolviendo una lista de diccionarios con los chunks y metadatos asociados a cada documento.
    """
    corpus = []
    for doc_path in Path(corpus_dir).rglob("*.md"):
        doc_chunks = chunking_documento(doc_path)
        corpus.append(doc_chunks)
    return corpus


def generar_embeddings_corpus(chunks_corpus, modelo_emb):
    """
    Genera embeddings para todos los chunks del corpus y preserva metadata.
    """
    flat_chunks = []

    # Aplanar manteniendo metadata
    for doc in chunks_corpus:
        for chunk in doc["chunks"]:
            flat_chunks.append({
                "chunk": chunk,
                "archivo": doc["archivo"],
                "categoria": doc["categoria"],
                "empresa": doc["empresa"]
            })

    # Preparar textos para E5
    chunks_e5 = ["passage: " + c["chunk"] for c in flat_chunks]

    # Embeddings en batch
    embeddings = modelo_emb.encode(
        chunks_e5,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    # Unir embeddings + metadata
    embeddings_metadata = []
    for i, item in enumerate(flat_chunks):
        embeddings_metadata.append({
            "embedding": embeddings[i],
            "chunk": item["chunk"],
            "archivo": item["archivo"],
            "categoria": item["categoria"],
            "empresa": item["empresa"]
        })

    return embeddings_metadata

c:\Users\andre\Documents\GitHub\PECL_LLMs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding

modelo_emb = SentenceTransformer("intfloat/multilingual-e5-base")
sparse_model = SparseTextEmbedding("Qdrant/bm25")
corpus_dir = "corpus/"

# Chunking + embeddings para todo el corpus
chunks_corpus = chunking_corpus(corpus_dir)
embeddings_corpus = generar_embeddings_corpus(chunks_corpus, modelo_emb)

# Sparse vectors para busqueda BM25
textos = [item["chunk"] for item in embeddings_corpus]
sparse_vectors = list(sparse_model.embed(textos))

print(f"Documentos procesados: {len(chunks_corpus)}")
print(f"Chunks con embedding: {len(embeddings_corpus)}")
print(f"Vectores sparse generados: {len(sparse_vectors)}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3814.76it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]c:\Users\andre\Documents\GitHub\PECL_LLMs\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\andre\AppData\Local\Temp\fastembed_cache\models--Qdrant--bm25. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggi

Documentos procesados: 6
Chunks con embedding: 268
Vectores sparse generados: 268


### 3.2. Base de datos vectorial Qdrant

Levantar el servidor Qdrant con Docker

PowerShell (Windows):

```powershell
docker run -p 6333:6333 -p 6334:6334 -v ${PWD}/qdrant_storage:/qdrant/storage qdrant/qdrant
```

Bash (Linux/macOS):

```bash
docker run -p 6333:6333 -p 6334:6334 -v "$(pwd)/qdrant_storage:/qdrant/storage" qdrant/qdrant
```

In [13]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, SparseVectorParams, PointStruct, SparseVector

# Conectar con Qdrant
cliente = QdrantClient(host="localhost", port=6333)

collection_name = "rag_privacy_corpus"

# Recrear coleccion para evitar conflicto de esquema o ids previos
if cliente.collection_exists(collection_name=collection_name):
    cliente.delete_collection(collection_name=collection_name)

cliente.create_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": VectorParams(size=768, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)

print(f"Coleccion lista: {collection_name}")

Coleccion lista: rag_privacy_corpus


In [14]:
# Ingest: insertar los embeddings con metadata en Qdrant
puntos = []
for i, item in enumerate(embeddings_corpus):
    puntos.append(
        PointStruct(
            id=i,
            vector={
                "dense": item["embedding"].tolist(),
                "sparse": SparseVector(
                    indices=sparse_vectors[i].indices.tolist(),
                    values=sparse_vectors[i].values.tolist(),
                ),
            },
            payload={
                "chunk": item["chunk"],
                "archivo": item["archivo"],
                "categoria": item["categoria"],
                "empresa": item["empresa"],
            },
        )
    )

# Upsert: inserta o actualiza si el id ya existe
cliente.upsert(collection_name="rag_privacy_corpus", points=puntos)
print(f"Puntos insertados/actualizados: {len(puntos)}")

Puntos insertados/actualizados: 268


In [25]:
from qdrant_client import models
from qdrant_client.models import Filter, FieldCondition, MatchValue
from sentence_transformers import CrossEncoder

# Inicializar el cross-encoder para reranking
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def buscar_qdrant(query, top_k_inicial=10, top_k_final=3, filtro_categoria=None, filtro_empresa=None):
    """
    Realiza una búsqueda en Qdrant utilizando un pipeline completo de RAG:

    1. Búsqueda híbrida (dense + sparse/BM25):
       - Combina embeddings densos (E5) con representaciones sparse (BM25)
       - Los resultados de ambas búsquedas se fusionan mediante Reciprocal Rank Fusion (RRF)
       - Mejora la recuperación al capturar similitud semántica y coincidencias léxicas

    2. Filtrado por metadatos (opcional):
       - Permite restringir resultados por categoría o empresa

    3. Reranking con cross-encoder:
       - Reordena los resultados iniciales evaluando directamente la relación (query, chunk)
       - Mejora la precisión final del contexto

    Parameters:
        query (str): consulta del usuario
        top_k_inicial (int): número de candidatos recuperados inicialmente
        top_k_final (int): número de resultados tras reranking
        filtro_categoria (str, opcional): filtrar por categoría
        filtro_empresa (str, opcional): filtrar por empresa

    Returns:
        list[dict]: lista de chunks relevantes con metadata y scores
    """
    # --- 1. RETRIEVAL HÍBRIDO (dense + sparse)
    # Embedding denso (semántico)
    query_emb = modelo_emb.encode(["query: " + query], normalize_embeddings=True)[0]
    
    # Embedding disperso (coincidencia léxica)
    query_sparse_raw = list(sparse_model.embed([query]))[0]
    query_sparse = SparseVector(
        indices=query_sparse_raw.indices.tolist(),
        values=query_sparse_raw.values.tolist(),
    )

    # FILTRADO POR METADATA (opcional)
    filtro = None
    if filtro_categoria or filtro_empresa:
        must_conditions = []
        
        if filtro_categoria:
            must_conditions.append(
                FieldCondition(
                    key="categoria",
                    match=MatchValue(value=filtro_categoria),
                )
            )

        if filtro_empresa:
            must_conditions.append(
                FieldCondition(
                    key="empresa",
                    match=MatchValue(value=filtro_empresa),
                )
            )

        filtro = Filter(must=must_conditions)

    # --- BÚSQUEDA EN QDRANT (HÍBRIDA)
    # Qdrant combina automáticamente dense + sparse en un único score híbrido
    resultados = cliente.query_points(
        collection_name="rag_privacy_corpus",
        prefetch=[
            # Recupera candidatos de forma independiente para cada tipo de representación
            models.Prefetch(
                query=query_emb.tolist(),
                using="dense",
                limit=top_k_inicial * 2, # Doble de candidatos para mejorar recall antes del reranking
            ),
            models.Prefetch(
                query=query_sparse,
                using="sparse",
                limit=top_k_inicial * 2,
            ),
        ],
        # Combina los rankings de dense y sparse usando RRF
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        # Se reduce el número de candidatos antes del reranking
        limit=top_k_inicial, 
        query_filter=filtro,
        with_payload=True,
    )

    # --- 2. RERANKING: el cross-encoder evalúa cada par (query, chunk) conjuntamente
    # Crear pares (query, chunk)
    pairs = [
        (query, punto.payload["chunk"])
        for punto in resultados.points
    ]

    # Obtener scores del reranker
    scores = reranker.predict(pairs)

    # Combinar resultados
    reranked = []
    for i, punto in enumerate(resultados.points):
        reranked.append({
            "chunk": punto.payload["chunk"],
            "categoria": punto.payload["categoria"],
            "archivo": punto.payload["archivo"],
            "empresa": punto.payload.get("empresa", None),
            # Score híbrido (dense + sparse combinado por Qdrant)
            "score_hybrid": punto.score,
            # Score del reranker (decisión final)
            "score_rerank": float(scores[i])
        })

    # Ordenar por score del reranker
    reranked = sorted(reranked, key=lambda x: x["score_rerank"], reverse=True)

    # Seleccionar top final
    reranked_final = reranked[:top_k_final]

    # --- DEBUG / PRINT
    print(f"Query: '{query}'")
    if filtro_categoria:
        print(f"Filtro: categoria = '{filtro_categoria}'")
    if filtro_empresa:
        print(f"Filtro: empresa = '{filtro_empresa}'")
    print()

    for i, r in enumerate(reranked_final):
        print(f"{i + 1}. (rerank={r['score_rerank']:.3f} | hybrid={r['score_hybrid']:.3f}) [{r['categoria']} - {r['archivo']}]")
        print(f"   {r['chunk']}\n")

    return reranked_final

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5410.23it/s]


In [26]:
# Búsqueda sin filtro
_ = buscar_qdrant("¿Qué datos personales recopila Google?")

Query: '¿Qué datos personales recopila Google?'

1. (rerank=0.725 | hybrid=0.200) [privacy_policies - google_privacy.md]
   When you’re signed in, we also collect information that we store with your Google Account, which we treat aspersonal information.

2. (rerank=0.294 | hybrid=0.312) [privacy_policies - spotify_privacy.md]
   | Datos de pago y compra | Si compras algo en Spotify o te registras en una Opción de Servicio de pago o a una prueba, necesitaremos procesar tus datos de pago. Los datos personales exactos recopilados y utilizados variarán en función de la forma de pago, pero incluirán información como la siguiente: Nombre Fecha de nacimiento Tipo de forma de pago (por ejemplo, tarjeta de crédito o débito) Si utilizas una tarjeta de débito o crédito, el tipo de tarjeta, la fecha de vencimiento y determinados dígitos del número de tu tarjeta. (Nota: Por motivos de seguridad, nunca almacenamos tu número de tarjeta completo). Código postal Número de teléfono móvil Información de 

In [27]:
# Búsqueda con filtro por categoría
_ = buscar_qdrant("¿Qué datos personales recopila Google?", filtro_categoria="privacy_policies")

Query: '¿Qué datos personales recopila Google?'
Filtro: categoria = 'privacy_policies'

1. (rerank=0.725 | hybrid=0.200) [privacy_policies - google_privacy.md]
   When you’re signed in, we also collect information that we store with your Google Account, which we treat aspersonal information.

2. (rerank=0.294 | hybrid=0.317) [privacy_policies - spotify_privacy.md]
   | Datos de pago y compra | Si compras algo en Spotify o te registras en una Opción de Servicio de pago o a una prueba, necesitaremos procesar tus datos de pago. Los datos personales exactos recopilados y utilizados variarán en función de la forma de pago, pero incluirán información como la siguiente: Nombre Fecha de nacimiento Tipo de forma de pago (por ejemplo, tarjeta de crédito o débito) Si utilizas una tarjeta de débito o crédito, el tipo de tarjeta, la fecha de vencimiento y determinados dígitos del número de tu tarjeta. (Nota: Por motivos de seguridad, nunca almacenamos tu número de tarjeta completo). Código postal N

In [28]:
# Búsqueda con filtro por categoría + empresa
_ = buscar_qdrant("¿Qué datos personales recopila Google?", filtro_categoria="privacy_policies", filtro_empresa="Google")
# Con ambos filtros, se consiguen resultados sean más precisos y relevantes

Query: '¿Qué datos personales recopila Google?'
Filtro: categoria = 'privacy_policies'
Filtro: empresa = 'Google'

1. (rerank=0.725 | hybrid=0.500) [privacy_policies - google_privacy.md]
   When you’re signed in, we also collect information that we store with your Google Account, which we treat aspersonal information.

2. (rerank=0.689 | hybrid=0.333) [privacy_policies - google_privacy.md]
   In some circumstances, Google also collects information about you frompublicly accessible sources. For example, if your name appears in your local newspaper, Google’s Search engine may index that article and display it to other people if they search for your name. We may also collect information about you from trusted partners, such as directory services who provide us with business information to be displayed on Google’s services, marketing partners who provide us with information about potential customers of our business services, and security partners who provide us with information toprotect a

In [29]:
# Pregunta fuera del dominio 
_ = buscar_qdrant("¿Cómo funciona la fotosíntesis en las plantas?")
# Devuelve chunks, pero el reranking deja el score a 0, mostrando que no son relevantes

Query: '¿Cómo funciona la fotosíntesis en las plantas?'

1. (rerank=0.000 | hybrid=0.250) [terms - spotify_terms.md]
   . POR OTRA PARTE, SPOTIFY NO HACE NINGUNA DECLARACIÓN, NI GARANTIZA NI ASUME RESPONSABILIDAD ALGUNA, EN RELACIÓN CON LAS APLICACIONES DE TERCEROS (O EL CONTENIDO DE LAS MISMAS), LOS CONTENIDOS DE LOS USUARIOS, LOS DISPOSITIVOS O CUALQUIER PRODUCTO O SERVICIO ANUNCIADO, PROMOCIONADO U OFRECIDO POR UN TERCERO EN EL SERVICIO O EN CUALQUIER SITIO WEB CON HIPERVÍNCULOS, O A TRAVÉS DE LOS MISMOS, Y SPOTIFY NO ES RESPONSABLE DE LAS TRANSACCIONES ENTRE USTED Y LOS PROVEEDORES EXTERNOS DE LO QUE ANTECEDE. NINGÚN CONSEJO NI INFORMACIÓN, VERBAL O POR ESCRITO, QUE USTED OBTENGA DE SPOTIFY DARÁ LUGAR A NINGUNA GARANTÍA EN NOMBRE DE SPOTIFY. DURANTE LA UTILIZACIÓN DEL SERVICIO DE SPOTIFY, ES POSIBLE QUE TENGA ACCESO A LAS FUNCIONES DE FILTRADO DE CONTENIDOS EXPLÍCITOS, PERO EL USO DE DICHAS FUNCIONES PODRÁ DAR LUGAR A QUE SE OFREZCA ALGÚN CONTENIDO EXPLÍCITO, POR LO QUE NO DEBERÁ C